In [1]:
import asyncio
import random

# Initialize the Asynchronous State Bridge Queue
# Maxsize=1 prevents old, stale frames from piling up if the VLM lags
state_bridge = asyncio.Queue(maxsize=1)

async def vlm_producer():
    """
    Simulates the cognitive VLM loop running at its own irregular speed.
    Generates new target coordinates based on visual data.
    """
    try:
        for i in range(1, 6):
            print(f"\n[VLM Producer] Step {i}: Analyzing camera frame...")
            # Simulate irregular cognitive thinking time (0.4 to 0.8 seconds)
            await asyncio.sleep(random.uniform(0.4, 0.8))

            target_coord = round(random.uniform(-10.0, 10.0), 2)
            print(f"[VLM Producer] New target matrix calculated: {target_coord}")

            # If the queue is full, drop the oldest stale frame to make room for the fresh one
            if state_bridge.full():
                try:
                    state_bridge.get_nowait()
                    print("[VLM Producer] QUEUE FULL: Dropped stale coordinate frame.")
                except asyncio.QueueEmpty:
                    pass

            # Put the fresh coordinate into the state bridge
            await state_bridge.put(target_coord)
            print(f"[VLM Producer] Deposited target {target_coord} into State Bridge.")

    except asyncio.CancelledError:
        print("[VLM Producer] Loop cleanly terminated.")

async def ik_consumer():
    """
    Simulates the physical Inverse Kinematics motor control loop.
    Consumes targets and updates the joints at its own fast, independent speed.
    """
    try:
        processed_count = 0
        while processed_count < 5:
            print("[IK Consumer] Checking State Bridge for targets...")

            # Non-blocking wait: suspends ONLY this coroutine if empty
            target = await state_bridge.get()
            processed_count += 1

            print(f" -> [IK Consumer] SUCCESS: Retrieved target {target} from Bridge.")
            print(f" -> [IK Consumer] Executing trajectory interpolation to {target}...")

            # Simulate immediate, high-frequency physical motor movement execution time
            await asyncio.sleep(0.1)
            print(f" -> [IK Consumer] Hardware reached target position: {target}.\n")

    except asyncio.CancelledError:
        print("[IK Consumer] Motor tracking cleanly terminated.")

async def run_pipeline():
    print("--- STARTING ASYNC PRODUCER-CONSUMER PIPELINE ---")
    # Run both loops concurrently using gather
    await asyncio.gather(vlm_producer(), ik_consumer())
    print("--- PIPELINE EXPERIMENT COMPLETE ---")

# Execute the runtime cell
await run_pipeline()

--- STARTING ASYNC PRODUCER-CONSUMER PIPELINE ---

[VLM Producer] Step 1: Analyzing camera frame...
[IK Consumer] Checking State Bridge for targets...
[VLM Producer] New target matrix calculated: -9.0
[VLM Producer] Deposited target -9.0 into State Bridge.

[VLM Producer] Step 2: Analyzing camera frame...
 -> [IK Consumer] SUCCESS: Retrieved target -9.0 from Bridge.
 -> [IK Consumer] Executing trajectory interpolation to -9.0...
 -> [IK Consumer] Hardware reached target position: -9.0.

[IK Consumer] Checking State Bridge for targets...
[VLM Producer] New target matrix calculated: -4.1
[VLM Producer] Deposited target -4.1 into State Bridge.

[VLM Producer] Step 3: Analyzing camera frame...
 -> [IK Consumer] SUCCESS: Retrieved target -4.1 from Bridge.
 -> [IK Consumer] Executing trajectory interpolation to -4.1...
 -> [IK Consumer] Hardware reached target position: -4.1.

[IK Consumer] Checking State Bridge for targets...
[VLM Producer] New target matrix calculated: 1.36
[VLM Producer] 